# 소나타 원본 PDF EDA\n\n대상 파일은 data/DN8_2026_ko_KR.pdf 하나다. 1 분석 목적, 2 대상 PDF, 3 PDF 로드, 4 페이지별 추출, 5 기본 통계, 6 빈 페이지·저텍스트 후보, 7 텍스트 분포, 8 Chapter, 9 DB 정합성, 10 결과 해석, 11 한계, 12 결론을 포함한다. 추출은 DocumentReader의 pypdf.PdfReader/page.extract_text() 흐름을 사용한다. Chapter는 CarManualRegisterService._extract_pdf_chapters와 동일한 level-1 PyMuPDF outline 규칙이다. DB 조회는 Sonata CAR_ID에 대한 SELECT뿐이다.

In [2]:

import os,re,json,statistics,unicodedata
from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
import pymupdf
from PIL import Image,ImageDraw,ImageFont
pdf=Path("data/DN8_2026_ko_KR.pdf");load_dotenv();reader=PdfReader(str(pdf),strict=False)
texts=[];errors=[]
for pno,p in enumerate(reader.pages,1):
 try:texts.append(p.extract_text())
 except Exception as e:texts.append(None);errors.append((pno,type(e).__name__,str(e)))
n=len(reader.pages);lengths=[len(t) if t is not None else 0 for t in texts]
nonblank=[i for i,t in enumerate(texts,1) if t is not None and bool(t.strip())]
none=[i for i,t in enumerate(texts,1) if t is None];empty=[i for i,t in enumerate(texts,1) if t==""];white=[i for i,t in enumerate(texts,1) if t is not None and t!="" and not t.strip()]
bins={"0":sum(x==0 for x in lengths),"1-99":sum(1<=x<=99 for x in lengths),"100-499":sum(100<=x<=499 for x in lengths),"500-999":sum(500<=x<=999 for x in lengths),"1000+":sum(x>=1000 for x in lengths)}
mean=statistics.mean(lengths);median=statistics.median(lengths);std=statistics.pstdev(lengths);short=[(i,x," ".join((texts[i-1] or "").split())) for i,x in enumerate(lengths,1) if x<100]
anoms=[]
for i,t in enumerate(texts,1):
 if "\ufffd" in t or "\x00" in t or re.search(r"([^\s.·…])\1{19,}",t):anoms.append(i)
 if "\\ufffd" in t or "\\x00" in t or re.search(r"([^\\s.·…])\\1{19,}",t):anoms.append(i)
pdfdoc=pymupdf.open(str(pdf));toc=[(title,start) for level,title,start in pdfdoc.get_toc(simple=True) if level==1]
chapters=[(title,start,toc[i+1][1]-1 if i+1<len(toc) else len(pdfdoc)) for i,(title,start) in enumerate(toc)];pdfdoc.close()
import psycopg
conn=psycopg.connect(os.environ["DB_URL"]);cur=conn.cursor();car_id="20261003_000014"
cur.execute("SELECT COUNT(*),MIN(car_manual_chunk_page_no),MAX(car_manual_chunk_page_no),COUNT(DISTINCT car_manual_chunk_page_no) FROM public.car_manual_chunk WHERE car_id=%s",(car_id,));db=cur.fetchone()
cur.execute("SELECT car_manual_chunk_page_no,COUNT(*) FROM public.car_manual_chunk WHERE car_id=%s GROUP BY car_manual_chunk_page_no ORDER BY car_manual_chunk_page_no",(car_id,));rows=cur.fetchall();cur.close();conn.close()
counts={p:c for p,c in rows if p is not None};missing=[p for p in range(1,n+1) if p not in counts];many=sorted([(p,c) for p,c in counts.items() if c>=5],key=lambda z:(-z[1],z[0]))
fig=Path("src/car_search_rag/car_search/docs/figures");fig.mkdir(parents=True,exist_ok=True);fp=Path("C:/Windows/Fonts/arial.ttf");f=ImageFont.truetype(str(fp),18) if fp.exists() else ImageFont.load_default();sf=ImageFont.truetype(str(fp),13) if fp.exists() else ImageFont.load_default()
def axes(w,h,title,xlab,ylab):
 im=Image.new("RGB",(w,h),"white");d=ImageDraw.Draw(im);d.text((60,20),title,fill="#172554",font=f);l,t,r,b=75,65,w-25,h-65;d.line((l,t,l,b),fill="#334155",width=2);d.line((l,b,r,b),fill="#334155",width=2);d.text((l,b+25),xlab,fill="#334155",font=sf);d.text((5,t),ylab,fill="#334155",font=sf);return im,d,(l,t,r,b)
edges=[0,100,200,300,400,500,750,1000,1500,2000,3000,5000,10000];hist=[sum(a<=v<(edges[i+1] if i+1<len(edges) else float("inf")) for v in lengths) for i,a in enumerate(edges)]
im,d,(l,t,r,b)=axes(920,520,"Sonata PDF: text length histogram","Characters/bin","Pages")
for i,v in enumerate(hist):
 w=(r-l)/len(hist);x=l+i*w+3;y=b-(b-t)*v/max(hist);d.rectangle((x,y,x+w-6,b),fill="#2563eb");d.text((x,b+4),str(edges[i]),fill="#334155",font=sf);d.text((x,y-18),str(v),fill="#0f172a",font=sf)
im.save(fig/"sonata_pdf_text_histogram.png")
for fn,title,vals,color,ylab in [("sonata_pdf_page_text_lengths.png","Sonata PDF: characters by page",lengths,"#2563eb","Characters"),("sonata_pdf_chunks_by_page.png","Sonata DB: chunks by page",[counts.get(p,0) for p in range(1,n+1)],"#059669","Chunks")]:
 im,d,(l,t,r,b)=axes(1400,560,title,"PDF page",ylab);mx=max(vals);d.line([(l+i*(r-l)/(n-1),b-v/mx*(b-t)) for i,v in enumerate(vals)],fill=color,width=2)
 for p in [1,100,200,300,400,n]:
  x=l+(p-1)*(r-l)/(n-1);d.text((x-10,b+8),str(p),fill="#334155",font=sf)
 im.save(fig/fn)
out={"pdf":str(pdf),"bytes":pdf.stat().st_size,"pages":n,"success":len(nonblank),"none":none,"empty":empty,"whitespace":white,"errors":errors,"mean":mean,"median":median,"min":min(lengths),"max":max(lengths),"std":std,"bins":bins,"short":short,"anomaly_candidate_pages":anoms,"chapters":chapters,"db":db,"missing_pages":missing,"pages_5plus":many}
print(json.dumps(out,ensure_ascii=False,indent=2))


{
  "pdf": "data\\DN8_2026_ko_KR.pdf",
  "bytes": 43423714,
  "pages": 508,
  "success": 508,
  "none": [],
  "empty": [],
  "whitespace": [],
  "errors": [],
  "mean": 1019.5492125984252,
  "median": 944.0,
  "min": 21,
  "max": 4180,
  "std": 582.3608613007665,
  "bins": {
    "0": 0,
    "1-99": 5,
    "100-499": 46,
    "500-999": 235,
    "1000+": 222
  },
  "short": [
    [
      4,
      21,
      "DN8_KO.book Page 4"
    ],
    [
      6,
      22,
      "DN8_KO.book Page ii"
    ],
    [
      108,
      22,
      "DN8_KO.book Page 24"
    ],
    [
      500,
      22,
      "DN8_KO.book Page 54"
    ],
    [
      501,
      26,
      "I 색인 DN8_KO.book Page 1"
    ]
  ],
  "anomaly_candidate_pages": [],
  "chapters": [
    [
      "1. 안내 및 차량 정보",
      7,
      26
    ],
    [
      "2. 안전 및 주의 사항",
      27,
      38
    ],
    [
      "3. 시트 및 안전 장치",
      39,
      84
    ],
    [
      "4. 클러스터",
      85,
      108
    ],
    [
      "5. 편의 장치",
      109,
      244
  

## 시각화 결과\n\n![페이지 길이 histogram](../src/car_search_rag/car_search/docs/figures/sonata_pdf_text_histogram.png)\n\nHistogram은 페이지별 추출 문자 수의 구간별 빈도를 보여준다.\n\n![페이지별 길이](../src/car_search_rag/car_search/docs/figures/sonata_pdf_page_text_lengths.png)\n\n페이지별 plot은 텍스트 추출 길이의 위치별 변동과 짧은 후보를 보여준다.\n\n![페이지별 chunk 수](../src/car_search_rag/car_search/docs/figures/sonata_pdf_chunks_by_page.png)\n\nChunk plot은 기존 DB의 page_no coverage와 페이지당 chunk 수 편차를 보여준다. 각 그래프는 오류를 확정하는 자료가 아니라 확인용이다.\n\n## 한계와 결론\n저텍스트 페이지는 시각 검토 후보로만 해석한다. DB coverage는 chunk page_no 존재 확인이며 원문의 완전한 보존을 증명하지 않는다.